## SpaceX Launch Records Dashboard


This notebook contains the interactive Plotly Dash application built for the IBM Data Science Capstone project. It allows dynamic filtering of launch sites and payload ranges to visualize SpaceX mission success rates.

To run a Dash application inside a Jupyter Notebook, we need to use the jupyter-dash functionality (which has been fully integrated into dash starting from version 2.11).

We only need to make two small modifications from Jupyterlite:

Initialize the app using JupyterDash or pass an inline parameter.

Tell the app.run command to display the dashboard inline inside the notebook cell using mode='inline'.

Here is the step by step analysis using plotly Dash



After visual analysis using the dashboard, you should be able to obtain some insights to answer the following five questions:

    - Which site has the largest successful launches?
    - Which site has the highest launch success rate?
    - Which payload range(s) has the highest launch success rate?
    - Which payload range(s) has the lowest launch success rate?
    - Which F9 Booster version (v1.0, v1.1, FT, B4, B5, etc.) has the highest launch success rate?


In [2]:
# Import required libraries
import pandas as pd
import dash
from dash import html
from dash import dcc
from dash.dependencies import Input, Output
import plotly.express as px

## Data Loading & Variable Initialization
Load the dataset containing the landing records and pull the boundary values for the payload constraints.

In [7]:
# Read the airline data into pandas dataframe
spacex_df = pd.read_csv("D:/Online Training/IBM Data Science Professional Certificate/Course 10. Data Science Capstone/Final Capestone Assignments/spacex_launch_dash.csv")
#spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv")
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()


Initialize Dash App Instance

In [9]:
# 3Initialize App for Jupyter Notebook
app = dash.Dash(__name__)

## App Layout Definition
Define the visual structure of the dashboard component tree including the dropdown filters, interactive range slider, pie chart, and scatter plot canvas.

In [10]:
# App Layout
app.layout = html.Div(children=[
    html.H1('SpaceX Launch Records Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    
    # TASK 1: Add a dropdown list to enable Launch Site selection
    dcc.Dropdown(
        id='site-dropdown',
        options=[
            {'label': 'All Sites', 'value': 'ALL'},
            *[{'label': site, 'value': site} for site in spacex_df['Launch Site'].unique()]
        ],
        value='ALL',
        placeholder="Select a Launch Site here",
        searchable=True
    ),
    html.Br(),

    # TASK 2: Add a pie chart to show the total successful launches count for all sites
    html.Div(dcc.Graph(id='success-pie-chart')),
    html.Br(),

    html.P("Payload range (Kg):"),
    
    # TASK 3: Add a Range Slider to select payload range
    dcc.RangeSlider(
        id='payload-slider',
        min=0, 
        max=10000, 
        step=1000,
        marks={i: f'{i} kg' for i in range(0, 10001, 2000)},
        value=[min_payload, max_payload]
    ),
    html.Br(),

    # TASK 4: Add a scatter chart to show the correlation between payload and launch success
    html.Div(dcc.Graph(id='success-payload-scatter-chart')),
])

## Callbacks
This section contains backend function calls that automatically listen to adjustments made on UI components (Dropdown and Slider) and re-render target graphs in real-time.

In [13]:
# 5. TASK 2 Callback: Update Pie Chart based on Dropdown Selection
@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def get_pie_chart(entered_site):
    if entered_site == 'ALL':
        fig = px.pie(
            spacex_df, 
            values='class', 
            names='Launch Site', 
            title='Total Success Launches By Site'
        )
    else:
        filtered_df = spacex_df[spacex_df['Launch Site'] == entered_site]
        site_counts = filtered_df['class'].value_counts().reset_index()
        site_counts.columns = ['class', 'count']
        
        fig = px.pie(
            site_counts, 
            values='count', 
            names='class', 
            title=f'Total Success Launches for site {entered_site}'
        )
    return fig


# 6. TASK 4 Callback: Update Scatter Plot based on Dropdown and Slider Selection
@app.callback(
    Output(component_id='success-payload-scatter-chart', component_property='figure'),
    [
        Input(component_id='site-dropdown', component_property='value'),
        Input(component_id='payload-slider', component_property='value')
    ]
)
def get_scatter_chart(entered_site, payload_range):
    low, high = payload_range
    mask = (spacex_df['Payload Mass (kg)'] >= low) & (spacex_df['Payload Mass (kg)'] <= high)
    filtered_df = spacex_df[mask]
    
    if entered_site == 'ALL':
        fig = px.scatter(
            filtered_df, x='Payload Mass (kg)', y='class',
            color='Launch Site', 
            title='Correlation between Payload and Success for all Sites'
        )
    else:
        site_df = filtered_df[filtered_df['Launch Site'] == entered_site]
        fig = px.scatter(
            site_df, x='Payload Mass (kg)', y='class',
            title=f'Correlation between Payload and Success for site {entered_site}'
        )
    
    return fig

## Run Application
Execute the line below to build the interface live within the notebook environment.

In [16]:
# 7. Run App Inline inside Jupyter
# Cell 12: Updated Code
if __name__ == '__main__':
    # Changed port from 8050 to 8052 to avoid conflicts
    app.run(mode='inline', port=8052)